In [ ]:
pip install spotipy

In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import pandas as pd
import time
import os

In [ ]:
df = pd.read_csv("../data/raw/spotify_songs.csv")
df = df.dropna()
df = df.drop_duplicates(subset='track_id', keep='first')
df = df.reset_index(drop=True)
df.shape

In [ ]:
# Set up Spotify API credentials (intentionnally deleted)
client_id = "..."
client_secret = "..."

sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(client_id=client_id,client_secret=client_secret))

In [ ]:
def get_track_popularity(track_id):
    try:
        track = sp.track(track_id)
        return track['popularity']
    except spotipy.SpotifyException as e:
        if e.http_status == 429:
            retry_after = int(e.headers.get('Retry-After', 5))
            print(f"Rate limit reached. Retrying after {retry_after}s...")
            time.sleep(retry_after)
            return get_track_popularity(track_id)
        else:
            print(f"Error on {track_id}: {e}")
            return None
    except Exception as e:
        print(f"Unexpected error for {track_id}: {e}")
        return None

In [ ]:
DAILY_LIMIT = 5000
DAY_NUMBER = 6    # Increased manually

OUTPUT_FILE = f"tracks_popularity_day{DAY_NUMBER}.csv"

start = (DAY_NUMBER - 1) * DAILY_LIMIT
end = DAY_NUMBER * DAILY_LIMIT

track_ids_to_update = df['track_id'].tolist()
track_ids_chunk = track_ids_to_update[start:end]

print(f"Processing tracks {start} to {end} (day {DAY_NUMBER})")

updated_info = []

for i, tid in enumerate(track_ids_chunk, 1):

    pop = get_track_popularity(tid)

    if pop is not None:
        updated_info.append({'track_id': tid, 'track_popularity': pop})


    # Autosave every 200 tracks
    if i % 200 == 0:
        pd.DataFrame(updated_info).to_csv(OUTPUT_FILE, index=False)
        print(f"Autosaved progress : {OUTPUT_FILE}")

pd.DataFrame(updated_info).to_csv(OUTPUT_FILE, index=False)
print(f"Saved final results for day {DAY_NUMBER} in {OUTPUT_FILE}")